In [1]:
import cv2
import os
from datetime import datetime

stream_url = "https://live.hdontap.com/hls/hosb1/hogan_holycross.stream/chunklist.m3u8?e=1772075624&eh=edge01.virginia.nginx.hdontap.com&t=bL0xLPUhL4CnZXbDjnJzVQ"
base_dir = r"C:\Users\mj2387\OneDrive - University of Arizona\ETC\Small studies\Holy_Cross_Pehnology_Livecam\Images"

cap = cv2.VideoCapture(stream_url)
success, frame = cap.read()

if success:
    now = datetime.now()
    year = now.strftime("%Y")
    month = now.strftime("%m")
    day = now.strftime("%d")
    
    save_path = os.path.join(base_dir, year, month, day)
    os.makedirs(save_path, exist_ok=True)
    
    timestamp = now.strftime("%Y%m%d_%H%M%S")
    filename = f"gcc_{timestamp}.jpg"
    full_file_path = os.path.join(save_path, filename)
    
    cv2.imwrite(full_file_path, frame)
    print(f"Saved {full_file_path}")
        
cap.release()

Saved C:\Users\mj2387\OneDrive - University of Arizona\ETC\Small studies\Holy_Cross_Pehnology_Livecam\Images\2026\02\25\gcc_20260225_090917.jpg


In [1]:
import cv2
import os
import time
from datetime import datetime

stream_url = "https://live.hdontap.com/hls/hosb1/hogan_holycross.stream/chunklist.m3u8?e=1772098855&eh=edge03.nginx.hdontap.com&t=MyjWmNrSHhAoivqhrcYdCA"

# Use the absolute path to force the exact save location
base_dir = r"C:\Users\mj2387\OneDrive - University of Arizona\ETC\Small studies\Holy_Cross_Pehnology_Livecam\Code\phenology_images"

cap = cv2.VideoCapture(stream_url)

# Try reading up to 5 times in case the stream is slow to open
success = False
for i in range(5):
    success, frame = cap.read()
    if success:
        break
    time.sleep(2)

if success:
    now = datetime.now()
    year = now.strftime("%Y")
    month = now.strftime("%m")
    day = now.strftime("%d")
    
    save_path = os.path.join(base_dir, year, month, day)
    os.makedirs(save_path, exist_ok=True)
    
    timestamp = now.strftime("%Y%m%d_%H%M%S")
    filename = f"gcc_{timestamp}.jpg"
    full_file_path = os.path.join(save_path, filename)
    
    cv2.imwrite(full_file_path, frame)
        
cap.release()

In [6]:
import os
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

page_url = "https://www.holycross.edu/visit/webcams" 
base_dir = r"C:\Users\mj2387\OneDrive - University of Arizona\ETC\Small studies\Holy_Cross_Pehnology_Livecam\Code\phenology_images"

options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.get(page_url)

time.sleep(15)

# Scroll down 400 pixels
driver.execute_script("window.scrollBy(0, 600);")
time.sleep(2)

now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

save_path = os.path.join(base_dir, year, month, day)
os.makedirs(save_path, exist_ok=True)

timestamp = now.strftime("%Y%m%d_%H%M%S")
filename = f"gcc_{timestamp}.png"
full_file_path = os.path.join(save_path, filename)

driver.save_screenshot(full_file_path)
driver.quit()

In [11]:
import os
import time
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from PIL import Image

page_url = "https://www.holycross.edu/visit/webcams" 
base_dir = r"C:\Users\mj2387\OneDrive - University of Arizona\ETC\Small studies\Holy_Cross_Pehnology_Livecam\Code\phenology_images"

# Adjust these decimal values to change the crop percentage (0.10 equals 10 percent)
crop_top = 0.07
crop_bottom = 0.03
crop_left = 0.11
crop_right = 0.12

options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)
driver.get(page_url)

time.sleep(15)
driver.execute_script("window.scrollBy(0, 600);")
time.sleep(2)

now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

save_path = os.path.join(base_dir, year, month, day)
os.makedirs(save_path, exist_ok=True)

timestamp = now.strftime("%Y%m%d_%H%M%S")
filename = f"gcc_{timestamp}.png"
full_file_path = os.path.join(save_path, filename)

driver.save_screenshot(full_file_path)
driver.quit()

# Crop the saved image
img = Image.open(full_file_path)
width, height = img.size

left = width * crop_left
top = height * crop_top
right = width * (1.0 - crop_right)
bottom = height * (1.0 - crop_bottom)

cropped_img = img.crop((left, top, right, bottom))
cropped_img.save(full_file_path)

In [6]:
%%writefile phenocam_archiver.py
import asyncio
import sys
import cv2
import os
from datetime import datetime
from playwright.async_api import async_playwright

# Fix for Windows asyncio loop when running outside Jupyter
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

async def get_live_m3u8(page_url):
    """Intercepts the HDOnTap network traffic to grab the live stream URL."""
    m3u8_url = None
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        def handle_response(response):
            nonlocal m3u8_url
            if "chunklist.m3u8" in response.url or "playlist.m3u8" in response.url:
                m3u8_url = response.url
                
        page.on("response", handle_response)
        
        print("Loading page and intercepting network traffic...")
        await page.goto(page_url)
        await page.wait_for_timeout(5000) 
        await browser.close()
        
    return m3u8_url

async def main():
    webcam_url = "https://hdontap.com/stream/178090/holy-cross-hogan-courtyard-live-webcam/"
    
    # 1. Get current date and time
    now = datetime.now()
    year_str = now.strftime('%Y')
    month_str = now.strftime('%m') # '01' to '12'
    
    # 2. Build the directory path: phenology_images/YYYY/MM
    # If you want this to save directly to your local Google Drive, 
    # change base_dir to your Drive path, e.g., r"C:\Users\mj2387\Google Drive\phenology_images"
    base_dir = "phenology_images"
    save_dir = os.path.join(base_dir, year_str, month_str)
    
    # Create the directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    
    # 3. Format the filename: HC1_YYYY_MM_DD_HH_MM.jpg
    filename = f"HC1_{now.strftime('%Y_%m_%d_%H_%M')}.jpg"
    filepath = os.path.join(save_dir, filename)

    # 4. Fetch the stream and capture the image
    fresh_link = await get_live_m3u8(webcam_url)
    
    if fresh_link:
        print("Stream intercepted. Extracting frame...")
        cap = cv2.VideoCapture(fresh_link)
        ret, frame = cap.read()
        cap.release()
        
        if ret:
            # Save the image to the constructed path
            cv2.imwrite(filepath, frame)
            print(f"Success! Image saved to:\n{filepath}")
        else:
            print("Failed to read frame from the video stream.")
    else:
        print("Could not intercept the stream link.")

if __name__ == "__main__":
    asyncio.run(main())

Writing phenocam_archiver.py


In [7]:
!python phenocam_archiver.py

Loading page and intercepting network traffic...
Stream intercepted. Extracting frame...
Success! Image saved to:
phenology_images\2026\06\HC1_2026_06_04_10_46.jpg


In [8]:
import cv2
import numpy as np

# Global variables to store the coordinates
points = []
image = None
clone = None

def draw_polygon(event, x, y, flags, param):
    global points, image

    # Listen for left mouse clicks
    if event == cv2.EVENT_LBUTTONDOWN:
        points.append((x, y))
        
        # Draw a small dot at the clicked point
        cv2.circle(image, (x, y), 3, (0, 255, 0), -1)
        
        # Draw a green line connecting to the previous point
        if len(points) >= 2:
            cv2.line(image, points[-2], points[-1], (0, 255, 0), 2)
        
        cv2.imshow("Draw Canopy ROI", image)

def main():
    global image, clone, points
    
    image_path = 'hogan_courtyard_reference.jpg'
    image = cv2.imread(image_path)
    
    if image is None:
        print(f"Error: Could not load {image_path}. Make sure the script is running in the correct folder.")
        return

    clone = image.copy()
    
    cv2.namedWindow("Draw Canopy ROI")
    cv2.setMouseCallback("Draw Canopy ROI", draw_polygon)

    print("\n--- ROI Drawing Instructions ---")
    print("1. LEFT CLICK along the edges of the tree canopies to draw your polygon.")
    print("2. Press 'c' to CLOSE the shape and SAVE the mask.")
    print("3. Press 'r' to RESET if you make a mistake.")
    print("4. Press 'q' to QUIT without saving.\n")

    while True:
        cv2.imshow("Draw Canopy ROI", image)
        key = cv2.waitKey(1) & 0xFF

        # Press 'r' to reset the drawing
        if key == ord("r"):
            image = clone.copy()
            points = []
            print("Canvas reset.")

        # Press 'c' to confirm and save
        elif key == ord("c"):
            if len(points) > 2:
                # Draw the final line to close the shape visually
                cv2.line(image, points[-1], points[0], (0, 255, 0), 2)
                cv2.imshow("Draw Canopy ROI", image)
                cv2.waitKey(500) 
                
                # Create the pure black mask
                mask = np.zeros(image.shape[:2], dtype=np.uint8)
                
                # Fill the drawn polygon area with pure white (255)
                pts = np.array(points, np.int32).reshape((-1, 1, 2))
                cv2.fillPoly(mask, [pts], 255)
                
                # Save it to disk
                cv2.imwrite("canopy_mask.png", mask)
                print("Success! Mask saved as 'canopy_mask.png'.")
                break
            else:
                print("You need at least 3 points to create a mask!")

        # Press 'q' to quit
        elif key == ord("q"):
            print("Operation cancelled.")
            break

    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


--- ROI Drawing Instructions ---
1. LEFT CLICK along the edges of the tree canopies to draw your polygon.
2. Press 'c' to CLOSE the shape and SAVE the mask.
3. Press 'r' to RESET if you make a mistake.
4. Press 'q' to QUIT without saving.

Success! Mask saved as 'canopy_mask.png'.


In [1]:
git init
git add .
git commit -m "Initial commit with logger, mask, and frontend"
git branch -M main
git remote add origin https://github.com/mostafajavadian/hogan-phenocam.git
git push -u origin main

SyntaxError: invalid syntax (905367375.py, line 1)